In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA ecom_poc;

-- 1. Gold Daily Sales & Cancellation Rate
-- Grain: 1 row per order_date
CREATE OR REPLACE TABLE gold_daily_sales
COMMENT 'Grain: 1 row per order_date. Net revenue counts only DELIVERED orders with at least one SUCCESS BDT transaction, deduplicated at the order level to prevent fan-out.'
AS
WITH successful_orders AS (
  SELECT DISTINCT order_id
  FROM silver_transactions
  WHERE txn_status = 'SUCCESS' AND currency = 'BDT'
)
SELECT
  o.order_date,
  COUNT(DISTINCT o.order_id) AS total_orders,
  COUNT(DISTINCT CASE WHEN o.order_status = 'DELIVERED' AND s.order_id IS NOT NULL THEN o.order_id END) AS paid_delivered_orders,
  COALESCE(SUM(CASE WHEN o.order_status = 'DELIVERED' AND s.order_id IS NOT NULL THEN o.order_total_bdt ELSE 0 END), 0) AS net_revenue_bdt,
  ROUND(COUNT(DISTINCT CASE WHEN o.order_status = 'CANCELLED' THEN o.order_id END) * 100.0 / COUNT(DISTINCT o.order_id), 2) AS cancellation_rate_pct
FROM silver_orders o
LEFT JOIN successful_orders s ON o.order_id = s.order_id
GROUP BY o.order_date;

-- 2. Gold Revenue Breakdown by Category & Country
-- Grain: 1 row per category + country
CREATE OR REPLACE TABLE gold_revenue_breakdown
COMMENT 'Grain: 1 row per category and country. Includes only DELIVERED orders with a verified SUCCESS BDT payment.'
AS
WITH successful_orders AS (
  SELECT DISTINCT order_id
  FROM silver_transactions
  WHERE txn_status = 'SUCCESS' AND currency = 'BDT'
)
SELECT
  o.category,
  o.country,
  COUNT(DISTINCT o.order_id) AS delivered_orders,
  SUM(o.quantity) AS total_units_sold,
  SUM(o.order_total_bdt) AS total_revenue_bdt
FROM silver_orders o
INNER JOIN successful_orders s ON o.order_id = s.order_id
WHERE o.order_status = 'DELIVERED'
GROUP BY o.category, o.country;

-- 3. Gold Payment Success Rate by Method
-- Grain: 1 row per payment_method
CREATE OR REPLACE TABLE gold_payment_success
COMMENT 'Grain: 1 row per payment_method across all valid Silver transactions.'
AS
SELECT
  payment_method,
  COUNT(txn_id) AS total_attempts,
  SUM(CASE WHEN txn_status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_txns,
  SUM(CASE WHEN txn_status = 'SUCCESS' THEN amount_bdt ELSE 0 END) AS processed_volume_bdt,
  ROUND(SUM(CASE WHEN txn_status = 'SUCCESS' THEN 1 ELSE 0 END) * 100.0 / COUNT(txn_id), 2) AS success_rate_pct
FROM silver_transactions
GROUP BY payment_method;

-- 4. Gold Top Customers by Spend
-- Grain: 1 row per customer_id
CREATE OR REPLACE TABLE gold_top_customers
COMMENT 'Grain: 1 row per customer_id ranked by delivered, paid spend in BDT.'
AS
WITH successful_orders AS (
  SELECT DISTINCT order_id
  FROM silver_transactions
  WHERE txn_status = 'SUCCESS' AND currency = 'BDT'
)
SELECT
  o.customer_id,
  COUNT(DISTINCT o.order_id) AS completed_orders,
  SUM(o.order_total_bdt) AS lifetime_spend_bdt,
  MAX(o.order_date) AS last_order_date
FROM silver_orders o
INNER JOIN successful_orders s ON o.order_id = s.order_id
WHERE o.order_status = 'DELIVERED'
GROUP BY o.customer_id
ORDER BY lifetime_spend_bdt DESC
LIMIT 100;

num_affected_rows,num_inserted_rows


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA ecom_poc;

-- Baseline Total Revenue (must match AI/BI & Power BI)
SELECT SUM(net_revenue_bdt) AS baseline_total_revenue_bdt FROM gold_daily_sales;

-- Row counts & sample outputs
SELECT 'gold_daily_sales' AS mart, COUNT(*) AS rows FROM gold_daily_sales
UNION ALL SELECT 'gold_revenue_breakdown', COUNT(*) FROM gold_revenue_breakdown
UNION ALL SELECT 'gold_payment_success', COUNT(*) FROM gold_payment_success
UNION ALL SELECT 'gold_top_customers', COUNT(*) FROM gold_top_customers;

SELECT * FROM gold_payment_success ORDER BY success_rate_pct DESC;
SELECT * FROM gold_revenue_breakdown;

category,country,delivered_orders,total_units_sold,total_revenue_bdt
Groceries,Bangladesh,169635,847992,5135849564.82
Electronics,Bangladesh,63134,315811,1917424806.63
Home & Kitchen,Bangladesh,51133,256913,1560964987.66
Fashion,Bangladesh,106243,531294,3223082893.22
Health & Beauty,Bangladesh,33569,167511,1019138481.15


In [0]:
%sql
USE workspace.ecom_poc;

SELECT
  SUM(net_revenue_bdt) AS baseline_total_revenue_bdt,
  SUM(paid_delivered_orders) AS total_paid_delivered_orders,
  SUM(total_orders) AS total_valid_orders
FROM gold_daily_sales;

baseline_total_revenue_bdt,total_paid_delivered_orders,total_valid_orders
12856460733.48,423714,983784
